# Dog Bounding Box Detection
Uses a pre-trained YOLOv5 model to detect dogs and draw bounding boxes.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2
import numpy as np
from audio_utils import load_data, spectrogram, plot_spectrogram
from scipy.optimize import linear_sum_assignment
import os
import csv
from pathlib import Path
import subprocess
import warnings
import torch
import torchreid
import torchvision.transforms as T
from scenedetect import open_video, SceneManager
from scenedetect.detectors import AdaptiveDetector
import json
import numpy as np
import pandas as pd
import deeplabcut
import tensorflow as tf
from dlclibrary import download_huggingface_model

download_huggingface_model("full_dog", "dog_model")

### Methods

In [ ]:
class AppearanceExtractor:
     # Standard ReID pre-processing
    _transform = T.Compose([
        T.ToPILImage(),
        T.Resize((256, 128)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])

    def __init__(self, model_name='osnet_x0_25', device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

        self.device = torch.device(device)

        self.model = torchreid.models.build_model(
            model_name,
            num_classes=1,
            pretrained=True
        )

        self.model.eval()
        self.model.to(self.device)
    
    @torch.no_grad() 
    def __call__(self, frame_bgr, boxes):
        """
        frame_bgr: HxWx3 BGR image
        boxes: list of [x1, y1, x2, y2] bounding boxes

        Returns: Nx512 tensor of appearance features
        """
        if not boxes:
            return torch.empty((0, 512), dtype=torch.float32)
        
        h, w = frame_bgr.shape[:2]
        tensors = []

        for box in boxes:
            x1, y1, x2, y2 = box

            x1 = int(max(0, x1))
            y1 = int(max(0, y1))
            x2 = int(min(w, x2))
            y2 = int(min(h, y2))


            if x2 <= x1 or y2 <= y1:
                tensors.append(torch.zeros((3, 256, 128), dtype=torch.float32))  # Empty tensor for invalid box
                continue

            crop = frame_bgr[y1:y2, x1:x2]
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            tensors.append(self._transform(crop_rgb))

        batch = torch.stack(tensors).to(self.device)
        embeddings = self.model(batch)

        norms = torch.norm(embeddings, dim=1, keepdim=True).clamp(min=1e-6)
        embeddings = (embeddings / norms).cpu().numpy()

        return embeddings.astype(np.float32)

In [ ]:
# Calculate mouth openness across a slice of the video
def mouth_openness(slice):
    individuals = slice.columns.get_level_values("individuals").unique()
    results = {}

    for ind in individuals:
        df = slice.xs(ind, level="individuals", axis=1).droplevel(0, axis=1)

        def pt(name):
            return np.array([
                df[(name, "x")].values,
                df[(name, "y")].values
            ]).T

        uj = pt("upper_jaw")
        lj = pt("lower_jaw")

        openness = np.linalg.norm(uj - lj, axis=1)

        xs = df.xs("x", level="coords", axis=1)
        ys = df.xs("y", level="coords", axis=1)

        left = xs.min().min()
        right = xs.max().max()
        top = ys.min().min()
        bottom = ys.max().max()

        results[ind] = {
            "mean": openness.mean(),
            "activity": np.mean(np.abs(np.gradient(openness))),
            "instability": openness.std(),
            "left": left,
            "right": right,
            "top": top,
            "bottom": bottom
        }

    return results

# Slice pose data for a specific timestamp
def cut_dataframe(df, fps, timestamp):
    frame_id = int(timestamp * fps)

    start = max(0, frame_id - 10)
    end = frame_id + 10

    slice = df.iloc[start:end]
    return slice

# Calculate IoU of two bounding boxes
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, ix2, iy1, iy2 = max(ax1, bx1), min(ax2, bx2), max(ay1, by1), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)

    intersection = (iw * ih) if iw > 0 and ih > 0 else 0
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - intersection

    return intersection / union if union > 0 else 0

# Generate bounding box color
def get_color(stable_id):
    box_colors = [
        (255, 0, 0),
        (0, 0, 255),
        (255, 255, 0),
        (255, 0, 255),
        (0, 255, 255),
        (128, 0, 255),
        (255, 128, 0),
    ]
    
    return box_colors[stable_id % len(box_colors)]

# Collect bark events
def collect_bark_events(video_path):
    s, framerate = load_data(video_path)

    freqs, times, power, log_spec = spectrogram(video_path, s, framerate)

    low_f = 300
    high_f = 3000

    band_mask = (freqs >= low_f) & (freqs <= high_f)
    band_energy = log_spec[band_mask].mean(axis=0)

    energy = (band_energy - np.mean(band_energy)) / (np.std(band_energy) + 1e-6)

    threshold = 1
    bark_frames = np.where(energy > threshold)[0]

    bark_events = []
    current = []

    for idx in bark_frames:
        if not current or idx - current[-1] <= 2:
            current.append(idx)
        else:
            bark_events.append(current)
            current = [idx]

    if current:
        bark_events.append(current)

    bark_spec_ids = [int(np.mean(event)) for event in bark_events]
    return times, log_spec, bark_spec_ids

# Helper to get spectrogram column for time t
def get_spec_col(times, time):
    return np.argmin(np.abs(times - time))

# Returns the most likely barking dog for a set of MO results
def mo_to_bounding_box(mo_results, stable_results):
    if len(mo_results) == 0 or len(stable_results) == 0:
        return None, None, None

    activities = []
    instabilities = []
    inds = []

    for ind, scores in mo_results.items():
        inds.append(ind)
        activities.append(scores["activity"])
        instabilities.append(scores["instability"])

    activities = np.array(activities)
    instabilities = np.array(instabilities)

    a = (activities - np.mean(activities)) / (np.std(activities) + 1e-6)
    s = (instabilities - np.mean(instabilities)) / (np.std(instabilities) + 1e-6)

    scores = 0.5 * a + 0.5 * s

    idx = np.argmax(scores)
    best_ind = inds[idx]
    best_score = scores[idx]

    print(f"- Best MO result: {best_ind}, Score: {best_score}")

    ind = best_ind if best_score > 0.5 else None
        
    if ind is not None:
        best = mo_results[ind]
    else:
        return None

    # Find the closest overlapping stable box
    mouth_box = np.array([
        best["left"],
        best["top"],
        best["right"],
        best["bottom"]
    ])

    best_iou = -1
    best_stable_id = None

    for stable_box, stable_id in stable_results:
        iou = box_iou(mouth_box, stable_box)

        if iou > best_iou:
            best_iou = iou
            best_stable_id = stable_id

    return best_stable_id

In [ ]:
# Track CSV export helpers
TRACK_COLUMNS = [
    "video_id", "frame", "time_sec", "track_id",
    "x1", "y1", "x2", "y2", "confidence",
]

def compute_frame_stride(fps, target_hz=10):
    fps = float(fps)
    if fps <= 0:
        return 1
    return max(1, round(fps / target_hz))

def build_track_rows(video_id, frame_id, fps, stable_results, stable_id_to_conf):
    rows = []
    time_sec = round(frame_id / float(fps), 2)

    for box, stable_id in stable_results:
        x1, y1, x2, y2 = map(int, box)
        rows.append({
            "video_id": video_id,
            "frame": frame_id,
            "time_sec": time_sec,
            "track_id": stable_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": stable_id_to_conf.get(stable_id, 0.0),
        })

    return rows

In [ ]:
# Detect transitions in the video
def detect_cut(self, frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if self.prev_gray is None:
        self.prev_gray = gray
        return False

    diff = np.mean(cv2.absdiff(gray, self.prev_gray))
    self.prev_gray = gray

    return diff > 40

# Stable tracker class with Kalman filter and combined IoU + appearance matching
def create_kalman_filter():
    kf = cv2.KalmanFilter(6, 4)

    kf.transitionMatrix = np.array([
        [1,0,0,0,1,0],
        [0,1,0,0,0,1],
        [0,0,1,0,0,1],
        [0,0,0,1,0,1],
        [0,0,0,0,1,0],
        [0,0,0,0,0,1],
    ], dtype=np.float32)

    kf.measurementMatrix = np.eye(4, 6, dtype=np.float32)
    kf.processNoiseCov = np.eye(6, dtype=np.float32) * 0.5
    kf.measurementNoiseCov = np.eye(4, dtype=np.float32) * 0.05
    kf.errorCovPost = np.eye(6, dtype=np.float32)

    return kf

def xyxy_to_xywh(box):
    x1, y1, x2, y2 = box
    x, y, w, h = (x1 + x2) / 2, (y1 + y2) / 2, (x2 - x1), (y2 - y1)
    return np.array([[x], [y], [w], [h]], dtype=np.float32)

class StableTracker:
    def __init__(self, iou_threshold, max_age, appearance_extractor, appearance_threshold, appearance_ema, iou_weight=0.6):
        self.next_stable_id = 0
        self.active = {}
        self.lost = {}
        self.yolo_to_stable = {}
        self.iou_threshold = iou_threshold
        self.max_age = max_age
        self.extractor = appearance_extractor
        self.appearance_threshold = appearance_threshold
        self.appearance_ema = appearance_ema
        self.iou_weight = iou_weight  # α: weight for IoU vs appearance in combined cost
        self.after_scene_change = 0
        self.colors = {}

    def cosine(self, a, b):
        return float(np.dot(a, b))

    def update_embedding(self, stable_id, new_embedding, store):
        alpha = self.appearance_ema
        old = store[stable_id].get('embedding')

        if old is None:
            store[stable_id]['embedding'] = new_embedding
        else:
            blended = alpha * new_embedding + (1 - alpha) * old
            norm = np.linalg.norm(blended)
            store[stable_id]['embedding'] = blended / norm if norm > 0 else blended

    def update(self, detections, frame_id, frame_bgr=None, scene_change=False):
        if scene_change:
            self.after_scene_change = 15  # Boost appearance matching for next few frames
            self.yolo_to_stable.clear()
        if self.extractor is not None and frame_bgr is not None and detections:
            boxes = [box for box, _ in detections]
            detection_embeddings = self.extractor(frame_bgr, boxes)
        else:
            detection_embeddings = [None] * len(detections)

        unmatched_detections = []
        unmatched_embeddings = []
        matched_stable_ids = set()

        # Stage 1: Match by YOLO ID — cheap, reliable, keep as-is
        for (box, yolo_id), emb in zip(detections, detection_embeddings):
            if yolo_id in self.yolo_to_stable:
                stable_id = self.yolo_to_stable[yolo_id]
                existing = self.active.get(stable_id)

                if existing is not None and box_iou(box, existing['box']) < 0.5:
                    del self.yolo_to_stable[yolo_id]
                    unmatched_detections.append((box, yolo_id))
                    unmatched_embeddings.append(emb)
                    continue

                if stable_id in self.lost:
                    self.active[stable_id] = self.lost.pop(stable_id)
                if stable_id in self.active:
                    kf = self.active[stable_id]['kf']
                    kf.correct(xyxy_to_xywh(box))
                    self.active[stable_id]['box'] = box
                    self.active[stable_id]['last_seen'] = frame_id
                    self.yolo_to_stable[yolo_id] = stable_id
                    if emb is not None:
                        self.update_embedding(stable_id, emb, self.active)
                    matched_stable_ids.add(stable_id)
            else:
                unmatched_detections.append((box, yolo_id))
                unmatched_embeddings.append(emb)

        # Stage 2: Combined IoU + appearance Hungarian match over all unmatched trackers
        if unmatched_detections:
            # Predict lost tracker positions
            pred_lost_boxes = {}
            for stable_id, data in self.lost.items():
                pred = data['kf'].predict()
                x, y, w, h = pred[:4].flatten()
                pred_lost_boxes[stable_id] = (x - w/2, y - h/2, x + w/2, y + h/2)

            # Candidates = all active (not yet matched) + all lost
            candidate_ids = (
                [sid for sid in self.active if sid not in matched_stable_ids] +
                list(self.lost.keys())
            )
            candidate_boxes = [
                pred_lost_boxes[sid] if sid in self.lost else self.active[sid]['box']
                for sid in candidate_ids
            ]
            candidate_store = {**self.active, **self.lost}

            if candidate_ids:
                α = self.iou_weight
                cost_matrix = np.ones((len(unmatched_detections), len(candidate_ids)), dtype=np.float32)

                for i, (box, _) in enumerate(unmatched_detections):
                    emb = unmatched_embeddings[i]
                    for j, (sid, cbox) in enumerate(zip(candidate_ids, candidate_boxes)):
                        iou_cost = 1 - box_iou(box, cbox)

                        cand_emb = candidate_store[sid].get('embedding')
                        if emb is not None and cand_emb is not None:
                            app_cost = 1 - self.cosine(emb, cand_emb)

                            if self.after_scene_change > 0:
                                cost_matrix[i, j] = app_cost
                            else:
                                cost_matrix[i, j] = α * iou_cost + (1 - α) * app_cost
                        else:
                            # No embeddings available — IoU only
                            cost_matrix[i, j] = iou_cost

                opt_det, opt_cand = linear_sum_assignment(cost_matrix)
                assignment = {det: cand for det, cand in zip(opt_det, opt_cand)}

                still_unmatched = []
                still_unmatched_embeddings = []

                for i, (box, yolo_id) in enumerate(unmatched_detections):
                    emb = unmatched_embeddings[i]

                    if i not in assignment:
                        still_unmatched.append((box, yolo_id))
                        still_unmatched_embeddings.append(emb)
                        continue

                    j = assignment[i]
                    sid = candidate_ids[j]
                    iou_score = box_iou(box, candidate_boxes[j])
                    cand_emb = candidate_store[sid].get('embedding')
                    app_score = self.cosine(emb, cand_emb) if emb is not None and cand_emb is not None else None

                    # Accept if IoU clears the threshold, or appearance alone is confident enough
                    if self.after_scene_change > 0:
                        accept = (
                            app_score is not None
                            and app_score >= self.appearance_threshold
                        )
                    else:
                        accept = (
                            iou_score >= self.iou_threshold
                            or (
                                app_score is not None
                                and app_score >= self.appearance_threshold
                            )
                        )

                    if accept:
                        if sid in self.lost:
                            self.active[sid] = self.lost.pop(sid)

                        prev_yolo_id = self.active[sid].get('yolo_id')
                        if prev_yolo_id and prev_yolo_id in self.yolo_to_stable:
                            del self.yolo_to_stable[prev_yolo_id]

                        kf = self.active[sid]['kf']
                        kf.correct(xyxy_to_xywh(box))
                        self.active[sid]['box'] = box
                        self.active[sid]['last_seen'] = frame_id
                        self.active[sid]['yolo_id'] = yolo_id
                        self.yolo_to_stable[yolo_id] = sid
                        if emb is not None:
                            self.update_embedding(sid, emb, self.active)
                        matched_stable_ids.add(sid)
                    else:
                        still_unmatched.append((box, yolo_id))
                        still_unmatched_embeddings.append(emb)

                unmatched_detections = still_unmatched
                unmatched_embeddings = still_unmatched_embeddings

        # Stage 3: Anything still unmatched becomes a new stable ID
        for (box, yolo_id), emb in zip(unmatched_detections, unmatched_embeddings):
            stable_id = self.next_stable_id
            self.next_stable_id += 1

            kf = create_kalman_filter()
            x, y, w, h = xyxy_to_xywh(box).flatten()
            kf.statePre = np.array([[x],[y],[w],[h],[0],[0]], np.float32)
            kf.statePost = kf.statePre.copy()

            self.yolo_to_stable[yolo_id] = stable_id
            self.active[stable_id] = {
                'kf': kf,
                'box': box,
                'last_seen': frame_id,
                'yolo_id': yolo_id,
                'embedding': emb
            }

        # Move aged-out trackers to lost
        for stable_id, data in list(self.active.items()):
            if frame_id - data['last_seen'] > self.max_age:
                self.lost[stable_id] = self.active.pop(stable_id)

        if self.after_scene_change > 0:
            self.after_scene_change -= 1

        # Return active stable IDs with their boxes
        return [(data['box'], sid) for sid, data in self.active.items()]

In [ ]:
# Processing function with built-in tracker (no Kalman or appearance)
def process_video_old(video_path, output_path, log_spec, bark_spec_ids, times, model, conf_threshold):
    # Load dog video
    cap = cv2.VideoCapture(video_path)

    print("Video opened:", cap.isOpened())

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Precompute spectrogram visualization
    spec_full = np.flipud(log_spec)

    spec_vis_full = cv2.normalize(spec_full, None, 0, 255, cv2.NORM_MINMAX)
    spec_vis_full = spec_vis_full.astype(np.uint8)
    spec_vis_full = cv2.applyColorMap(spec_vis_full, cv2.COLORMAP_INFERNO)
    spec_vis_full = cv2.resize(spec_vis_full, (w, h))

    # Video writer for output
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w * 2, h)
    )

    print("Writer opened:", out.isOpened())

    frame_id = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_id / fps # Current time in seconds
        spec_id = get_spec_col(times, t)
        spec_vis = spec_vis_full.copy()
        x_pos = int(spec_id / log_spec.shape[1] * w)
        cv2.line(spec_vis, (x_pos, 0), (x_pos, h), (0, 255, 255), 3)

        results = model.track(
            frame,
            persist=True,
            tracker="botsort.yaml",
            imgsz=640,
            conf=conf_threshold,
            verbose=False
        )

        r = results[0]

        if r.boxes is not None and r.boxes.id is not None:
            boxes = r.boxes.xyxy.cpu().numpy()
            cls_ids = r.boxes.cls.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()
            yolo_ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None

            for i in range(len(boxes)):
                if model.names[int(cls_ids[i])] == 'dog' and confs[i] > conf_threshold:
                    if yolo_ids is None:
                        continue

            # for box, stable_id in stable_results:
                x1, y1, x2, y2 = map(int, boxes[i])

                color = get_color(int(yolo_ids[i]))

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + 135, y1), color, -1)
                cv2.putText(frame, f'dog #{yolo_ids[i]}', 
                            (x1 + 5, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255, 255, 255), 2)

        for bark_id in bark_spec_ids:
            if abs(bark_id - spec_id) < 10:  # only nearby ones
                x = int(bark_id / log_spec.shape[1] * w)

                cv2.rectangle(
                    spec_vis,
                    (x - 5, 0),
                    (x + 5, h),
                    (255, 0, 255),
                    -1
                )

        frame = frame.astype('uint8')
        frame = np.ascontiguousarray(frame)

        frame = cv2.resize(frame, (w, h))
        spec_vis = cv2.resize(spec_vis, (w, h))
        frame = np.hstack([frame, spec_vis])

        out.write(frame)
        frame_id += 1

    cap.release()
    out.release()
    # cv2.destroyAllWindows()

    print("Processing complete")

In [ ]:
# Main processing function
def process_video(video_path, output_path, log_spec, bark_spec_ids, times, model, appearance_model, appearance_threshold, appearance_ema, conf_threshold, iou_threshold, max_age, tracks_csv_path=None, video_id=None, save_fps=10):
    # Load dog video
    cap = cv2.VideoCapture(video_path)

    print("Video opened:", cap.isOpened())

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    fps_float = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if tracks_csv_path is not None:
        if video_id is None:
            video_id = Path(video_path).stem
        track_rows = []
        frame_stride = compute_frame_stride(fps_float, save_fps)

    video = open_video(video_path)
    scene_manager = SceneManager()
    scene_manager.add_detector(AdaptiveDetector())

    scene_manager.detect_scenes(video)
    scenes = scene_manager.get_scene_list()

    scene_boundaries = set()
    for start, _ in scenes[1:]:
        scene_boundaries.add(start.get_frames())

    poses = deeplabcut.video_inference_superanimal(
        [video_path],
        "superanimal_quadruped",
        model_name="hrnet_w32",
        detector_name="fasterrcnn_resnet50_fpn_v2",
        dest_folder="files/dlc-outputs",
    )

    extractor = AppearanceExtractor(appearance_model)
    stable_tracker = StableTracker(iou_threshold, max_age, extractor, appearance_threshold, appearance_ema)

    print("Stable tracker initialized")

    # Precompute spectrogram visualization
    spec_full = np.flipud(log_spec)

    spec_vis_full = cv2.normalize(spec_full, None, 0, 255, cv2.NORM_MINMAX)
    spec_vis_full = spec_vis_full.astype(np.uint8)
    spec_vis_full = cv2.applyColorMap(spec_vis_full, cv2.COLORMAP_INFERNO)
    spec_vis_full = cv2.resize(spec_vis_full, (w, h))

    # Video writer for output
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w * 2, h)
    )

    print("Writer opened:", out.isOpened())

    frame_id = 0
    processed_barks = set()
    highlighted_dog = None
    highlight_until_frame = -1

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_id / fps # Current time in seconds
        spec_id = get_spec_col(times, t)
        spec_vis = spec_vis_full.copy()
        x_pos = int(spec_id / log_spec.shape[1] * w)
        cv2.line(spec_vis, (x_pos, 0), (x_pos, h), (0, 255, 255), 3)

        results = model.track(frame, persist=True, verbose=False)

        r = results[0]

        if r.boxes is not None:
            boxes = r.boxes.xyxy.cpu().numpy()
            cls_ids = r.boxes.cls.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()
            yolo_ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None

            detections = []
            conf_lookup = {}

            for i in range(len(boxes)):
                if model.names[int(cls_ids[i])] == 'dog' and confs[i] > conf_threshold:
                    if yolo_ids is None:
                        continue

                    conf_lookup[yolo_ids[i]] = confs[i]
                    detections.append((boxes[i], yolo_ids[i]))

            stable_results = stable_tracker.update(detections, frame_id, frame, scene_change=(frame_id in scene_boundaries))

            for bark_id in bark_spec_ids:
                if bark_id not in processed_barks and abs(spec_id - bark_id) <= 1:
                    bark_t = times[bark_id]

                    print(f"Processing bark at frame {bark_id}, time {bark_t}")
                    
                    x = int(bark_id / log_spec.shape[1] * w)

                    cv2.rectangle(
                        spec_vis,
                        (x - 5, 0),
                        (x + 5, h),
                        (255, 0, 255),
                        -1
                    )

                    slice = cut_dataframe(poses[video_path], fps, bark_t)
                    mo_results = mouth_openness(slice)

                    print(f"- MO Results: {mo_results}")

                    stable_id = mo_to_bounding_box(mo_results, stable_results)

                    if stable_id is not None:
                        highlighted_dog = stable_id
                        highlight_until_frame = frame_id + fps

                    processed_barks.add(bark_id)

            stable_id_to_conf = {}

            for yolo_id, stable_id in stable_tracker.yolo_to_stable.items():
                if yolo_id in conf_lookup:
                    stable_id_to_conf[stable_id] = conf_lookup[yolo_id]

            if tracks_csv_path is not None and frame_id % frame_stride == 0:
                track_rows.extend(
                    build_track_rows(video_id, frame_id, fps_float, stable_results, stable_id_to_conf)
                )

            for box, stable_id in stable_results:
                x1, y1, x2, y2 = map(int, box)
                conf = stable_id_to_conf.get(stable_id, 0)

                if stable_id == highlighted_dog and frame_id <= highlight_until_frame:
                    color = (0, 255, 0)
                else: 
                    color = get_color(stable_id)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + 135, y1), color, -1)
                cv2.putText(frame, f'dog #{stable_id} {conf:.2f}',
                            (x1 + 5, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255, 255, 255), 2)

        frame = frame.astype('uint8')
        frame = np.ascontiguousarray(frame)

        frame = cv2.resize(frame, (w, h))
        spec_vis = cv2.resize(spec_vis, (w, h))
        frame = np.hstack([frame, spec_vis])

        out.write(frame)
        frame_id += 1

    cap.release()
    out.release()
    # cv2.destroyAllWindows()

    if tracks_csv_path is not None:
        parent = os.path.dirname(tracks_csv_path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        with open(tracks_csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=TRACK_COLUMNS)
            writer.writeheader()
            writer.writerows(track_rows)
        print(f"Saved {len(track_rows)} track rows to {tracks_csv_path}")

    print("Processing complete")

### Models

In [ ]:
from ultralytics import YOLO, RTDETR

In [ ]:
video_folder = 'files/dataset'
output_folder = 'files/yolo8-outputs'

# Load YOLOv8 model
model = YOLO('yolov8n.pt')
model_name = 'yolo8'
appearance_model = 'osnet_x0_25'
appearance_threshold = 0.5
appearance_ema = 0.3
conf_threshold = 0.25
iou_threshold = 0.08
max_age = 20

# Load font
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 40)

In [ ]:
video_folder = 'files/dataset'
output_folder = 'files/rt-outputs'

# Load RT-DETR model
model = RTDETR('rtdetr-l.pt')
model_name = 'rt'
appearance_model = 'osnet_x0_25'
appearance_threshold = 0.7
appearance_ema = 0.3
conf_threshold = 0.6
iou_threshold = 0.08
max_age = 20

# Load font
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 40)

In [ ]:
video_folder = 'files/dataset'
output_folder = 'files/yolo26-outputs'

# Load YOLOv8 model
model = YOLO("yolo26n.pt")
model_name = 'yolo26'
appearance_model = 'osnet_x0_25'
appearance_threshold = 0.7
appearance_ema = 0.3
conf_threshold = 0.25
iou_threshold = 0.08
max_age = 20

# Load font
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 40)

In [ ]:
# Single video processing
warnings.filterwarnings("ignore")

title = 'dogs2'

video = f'files/{title}.mp4'
output = f'outputs/{title}/{title}-tracked.mp4'
tracks_csv = f'outputs/{title}/tracks.csv'

times, log_spec, bark_spec_ids = collect_bark_events(video)
process_video(video,
              output,
              log_spec, 
              bark_spec_ids, 
              times, 
              model, 
              appearance_model, 
              appearance_threshold, 
              appearance_ema, 
              conf_threshold, 
              iou_threshold, 
              max_age,
              tracks_csv_path=tracks_csv,
              video_id=title)

subprocess.run([
            "ffmpeg",
            "-y",
            "-i", output,
            "-i", video,
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest",
            f'outputs/{title}/{title}-final.mp4'
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
# Video processing loop
warnings.filterwarnings("ignore")

for video_path in os.listdir(video_folder):
    if video_path.endswith('.mp4'):
        video = os.path.join(video_folder, video_path)
        output = os.path.join(output_folder, video_path.replace('.mp4', '-output.mp4'))

        times, log_spec, bark_spec_ids = collect_bark_events(video)
        process_video(video, 
                      output, 
                      log_spec, 
                      bark_spec_ids, 
                      times, 
                      model, 
                      appearance_model, 
                      appearance_threshold, 
                      appearance_ema, 
                      conf_threshold, 
                      iou_threshold, 
                      max_age)

        subprocess.run([
            "ffmpeg",
            "-y",
            "-i", output,
            "-i", video,
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-c:v", "libx264",
            "-c:a", "aac",
            "-shortest",
            "-crf", "28",
            os.path.join(output_folder, 'finals', video_path.replace('.mp4', '-final.mp4'))
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


### Other

In [ ]:
from ultralytics import YOLO, RTDETR

In [ ]:
video_folder = 'files/data'
output_folder = 'files/rt-outputs'

# Load RT-DETR model
model = RTDETR('rtdetr-l.pt')
model_name = 'rt'
conf_threshold = 0.6

# Load font
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 40)

In [ ]:
# Single video processing
warnings.filterwarnings("ignore")

title = 'dogs3'

video = f'files/{title}.mp4'
output = f'files/{title}-{model_name}.mp4'

times, log_spec, bark_spec_ids = collect_bark_events(video)
process_video_old(video,
                  output,
                  log_spec, 
                  bark_spec_ids, 
                  times,
                  model,
                  conf_threshold)

subprocess.run([
            "ffmpeg",
            "-y",
            "-i", output,
            "-i", video,
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest",
            f'files/eval/{title}-{model_name}-final.mp4'
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
# Load dog image
img = Image.open('files/wynnie.jpg').convert('RGB')

# Run inference
results = model(img, device='cpu')


# Draw bounding boxes for dogs
draw = ImageDraw.Draw(img)
dog_count = 0

for result in results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]
        conf = float(box.conf[0])

        if label == 'dog' and conf > 0.5:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            draw.rectangle([x1, y1, x2, y2], outline='blue', width=4)
            draw.rectangle([x1, y1 - 45, x1 + 200, y1], fill='blue')
            draw.text((x1 + 4, y1 - 40), f'{label} {conf:.2f}', fill='white', font=font)

# Display result
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.show()